# March Machine Learning Mania 2026

## NCAA Basketball Tournament Prediction

**Competition:** [March Machine Learning Mania 2026](https://www.kaggle.com/competitions/march-machine-learning-mania-2026/overview/description)

### Goal
Predict the outcomes of every possible matchup in the NCAA Men's and Women's Basketball Tournaments. Submit win-probability predictions (0–1) for every pair of teams that could potentially meet in the 2026 tournament.

### Evaluation Metric
Submissions are evaluated using **log-loss** (binary cross-entropy):
$$\text{LogLoss} = -\frac{1}{N}\sum_{i=1}^{N}\left[y_i \log(\hat{p}_i) + (1-y_i)\log(1-\hat{p}_i)\right]$$

Lower is better. Clipping predictions to [0.05, 0.95] is standard practice to avoid extreme log-loss penalties.

### Submission Format
```
ID,Pred
2026_1101_1102,0.5
2026_1101_1103,0.7
...
```
The `ID` field is `Season_LowerTeamID_HigherTeamID`. `Pred` is the probability that the **lower-ID team wins**.

---
### Notebook Structure
1. Configuration & Imports
2. Data Loading & Schema Overview
3. Exploratory Data Analysis
4. Feature Engineering
5. Model Training (Logistic Regression, XGBoost, LightGBM)
6. Ensemble & Calibration
7. Submission Generation

## 0. Kaggle Setup & Data Download

Install the `kaggle` package and download the competition dataset via the Kaggle API.

**Prerequisites:**
1. Create a Kaggle account and accept the competition rules at https://www.kaggle.com/competitions/march-machine-learning-mania-2026
2. Generate an API token from https://www.kaggle.com/settings → Account → API → *Create New Token*
3. Place the downloaded `kaggle.json` in `~/.kaggle/kaggle.json` (Linux/Mac) or `%USERPROFILE%\.kaggle\kaggle.json` (Windows)


In [ ]:
# ── Install Kaggle API ──
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'kaggle'], check=True)
print("kaggle package ready.")

# ── Download competition data ──
import os
import zipfile

COMPETITION   = 'march-machine-learning-mania-2026'
DOWNLOAD_DIR  = os.path.join(os.getcwd(), 'data')
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Only download if data is not already present
if not any(f.endswith('.csv') for f in os.listdir(DOWNLOAD_DIR)):
    print(f"Downloading competition data to: {DOWNLOAD_DIR}")
    result = subprocess.run(
        ['kaggle', 'competitions', 'download', '-c', COMPETITION, '-p', DOWNLOAD_DIR],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("[ERROR]", result.stderr)
        print("Ensure ~/.kaggle/kaggle.json exists and you have accepted the competition rules.")
    else:
        # Unzip all downloaded archives
        for fname in os.listdir(DOWNLOAD_DIR):
            if fname.endswith('.zip'):
                zip_path = os.path.join(DOWNLOAD_DIR, fname)
                print(f"Extracting {fname} ...")
                with zipfile.ZipFile(zip_path, 'r') as zf:
                    zf.extractall(DOWNLOAD_DIR)
                os.remove(zip_path)
        print("Download and extraction complete.")
        print("Files:", sorted(os.listdir(DOWNLOAD_DIR)))
else:
    print(f"Data already present in {DOWNLOAD_DIR}:")
    print(sorted(f for f in os.listdir(DOWNLOAD_DIR) if f.endswith('.csv')))

# ── Set DATA_DIR for the rest of the notebook ──
# Override with '/kaggle/input/march-machine-learning-mania-2026' when running on Kaggle
DATA_DIR = DOWNLOAD_DIR if os.path.exists(DOWNLOAD_DIR) else '/kaggle/input/march-machine-learning-mania-2026'
print(f"\nDATA_DIR set to: {DATA_DIR}")


## 1. Configuration & Imports

In [ ]:
# ── Standard Library ──
import os
import warnings
import itertools

# ── Data / Numerical ──
import numpy as np
import pandas as pd

# ── Visualisation ──
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# ── Scikit-learn ──
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss
from sklearn.pipeline import Pipeline

# ── Gradient Boosting ──
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not available – install with: pip install xgboost")

try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False
    print("LightGBM not available – install with: pip install lightgbm")

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 50)

# ── Configuration ──
# DATA_DIR is set by the Kaggle download cell (Section 0).
# Override here if needed (e.g., running without the download cell):
try:
    DATA_DIR  # set by Section 0 download cell
except NameError:
    DATA_DIR = '/kaggle/input/march-machine-learning-mania-2026'
OUTPUT_DIR = '/kaggle/working'
SEASON     = 2026          # target tournament year
SEED       = 42
N_FOLDS    = 5
CLIP_LO    = 0.05          # prediction clipping bounds
CLIP_HI    = 0.95

# Gender prefixes present in the competition data
GENDERS = ['M', 'W']       # M = Men's, W = Women's

np.random.seed(SEED)
print(f"DATA_DIR  : {DATA_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"Target season: {SEASON}")

## 2. Data Loading & Schema Overview

### Key Files
| File | Description |
|------|-------------|
| `{M/W}Teams.csv` | Static team ID ↔ name mapping |
| `{M/W}Seasons.csv` | Season metadata (region names, day-zero date) |
| `{M/W}RegularSeasonCompactResults.csv` | Regular season game outcomes (compact) |
| `{M/W}RegularSeasonDetailedResults.csv` | Regular season with box-score stats |
| `{M/W}NCAATourneyCompactResults.csv` | Historical tournament outcomes (compact) |
| `{M/W}NCAATourneyDetailedResults.csv` | Historical tournament with box-score stats |
| `{M/W}NCAATourneySeeds.csv` | Tournament seeds per season |
| `{M/W}NCAATourneySlots.csv` | Tournament bracket structure |
| `MMasseyOrdinals.csv` | Third-party ranking systems (Men's only) |
| `{M/W}SampleSubmission.csv` | All possible 2026 matchup IDs |
| `Cities.csv` + `{M/W}GameCities.csv` | Game location data |

In [ ]:
def load_csv(name: str) -> pd.DataFrame:
    """Load a CSV from DATA_DIR, returning an empty DataFrame if not found."""
    path = os.path.join(DATA_DIR, name)
    if not os.path.exists(path):
        print(f"  [WARN] File not found: {name}")
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f"  Loaded {name:55s} → {df.shape}")
    return df


print("=== Men's Data ===")
m_teams          = load_csv('MTeams.csv')
m_seasons        = load_csv('MSeasons.csv')
m_reg_compact    = load_csv('MRegularSeasonCompactResults.csv')
m_reg_detailed   = load_csv('MRegularSeasonDetailedResults.csv')
m_tour_compact   = load_csv('MNCAATourneyCompactResults.csv')
m_tour_detailed  = load_csv('MNCAATourneyDetailedResults.csv')
m_seeds          = load_csv('MNCAATourneySeeds.csv')
m_slots          = load_csv('MNCAATourneySlots.csv')
m_massey         = load_csv('MMasseyOrdinals.csv')
m_sample_sub     = load_csv('MSampleSubmission.csv')

print("\n=== Women's Data ===")
w_teams          = load_csv('WTeams.csv')
w_seasons        = load_csv('WSeasons.csv')
w_reg_compact    = load_csv('WRegularSeasonCompactResults.csv')
w_reg_detailed   = load_csv('WRegularSeasonDetailedResults.csv')
w_tour_compact   = load_csv('WNCAATourneyCompactResults.csv')
w_tour_detailed  = load_csv('WNCAATourneyDetailedResults.csv')
w_seeds          = load_csv('WNCAATourneySeeds.csv')
w_slots          = load_csv('WNCAATourneySlots.csv')
w_sample_sub     = load_csv('WSampleSubmission.csv')

print("\n=== Shared Data ===")
cities           = load_csv('Cities.csv')
m_game_cities    = load_csv('MGameCities.csv')
w_game_cities    = load_csv('WGameCities.csv')

## 3. Exploratory Data Analysis

In [ ]:
# ── Schema preview ──
for name, df in [('m_reg_compact', m_reg_compact),
                  ('m_tour_compact', m_tour_compact),
                  ('m_seeds', m_seeds)]:
    if not df.empty:
        print(f"\n--- {name} ({df.shape}) ---")
        display(df.head(3))

In [ ]:
def season_summary(reg_df: pd.DataFrame, tour_df: pd.DataFrame, label: str):
    """Print basic season-level summary statistics."""
    if reg_df.empty:
        print(f"[{label}] No data available.")
        return
    seasons = sorted(reg_df['Season'].unique())
    print(f"[{label}] Regular-season games : {len(reg_df):,}  "
          f"| Seasons: {seasons[0]}–{seasons[-1]}  "
          f"| Teams per season (avg): {reg_df.groupby('Season')['WTeamID'].nunique().mean():.0f}")
    if not tour_df.empty:
        print(f"[{label}] Tournament games     : {len(tour_df):,}")

season_summary(m_reg_compact, m_tour_compact, "MEN")
season_summary(w_reg_compact, w_tour_compact, "WOMEN")

In [ ]:
# ── Score distributions ──
def plot_score_dist(reg_df: pd.DataFrame, label: str):
    if reg_df.empty:
        return
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Winning & losing score distributions
    axes[0].hist(reg_df['WScore'], bins=40, alpha=0.6, label='Winner', color='steelblue')
    axes[0].hist(reg_df['LScore'], bins=40, alpha=0.6, label='Loser',  color='salmon')
    axes[0].set_title(f'{label} – Score Distribution (Regular Season)')
    axes[0].set_xlabel('Points scored')
    axes[0].legend()

    # Point differential
    diff = reg_df['WScore'] - reg_df['LScore']
    axes[1].hist(diff, bins=40, color='mediumseagreen', edgecolor='white')
    axes[1].set_title(f'{label} – Point Differential (Winner − Loser)')
    axes[1].set_xlabel('Margin of victory')

    plt.tight_layout()
    plt.show()

plot_score_dist(m_reg_compact, "Men's")
plot_score_dist(w_reg_compact, "Women's")

In [ ]:
# ── Tournament upsets: seed difference vs win probability ──
def analyze_seeding_upsets(seeds_df: pd.DataFrame, tour_df: pd.DataFrame, label: str):
    if seeds_df.empty or tour_df.empty:
        return

    # Parse numeric seed (strip region letter and play-in markers)
    seeds_df = seeds_df.copy()
    seeds_df['SeedNum'] = seeds_df['Seed'].str.extract(r'(\d+)').astype(int)

    seed_map = seeds_df.set_index(['Season', 'TeamID'])['SeedNum']

    df = tour_df.copy()
    df['WSeed'] = df.set_index(['Season', 'WTeamID']).index.map(seed_map.to_dict().get)
    df['LSeed'] = df.set_index(['Season', 'LTeamID']).index.map(seed_map.to_dict().get)
    df = df.dropna(subset=['WSeed', 'LSeed'])
    df['SeedDiff'] = df['LSeed'] - df['WSeed']   # positive = favourite won

    upset_rate = (df.groupby('SeedDiff')
                    .apply(lambda x: (x['SeedDiff'] < 0).mean()))

    # Win rate of lower-seeded (better) team by matchup type
    matchup_wins = (df.groupby('SeedDiff').size() /
                    df.groupby('SeedDiff').size().sum())

    diff_counts = df['SeedDiff'].value_counts().sort_index()
    upset_mask  = diff_counts[diff_counts.index < 0]
    total_upsets = len(df[df['SeedDiff'] < 0])
    print(f"[{label}] Total tournament games : {len(df):,}")
    print(f"[{label}] Upsets (lower seed wins): {total_upsets:,}  "
          f"({100*total_upsets/len(df):.1f}%)")

    fig, ax = plt.subplots(figsize=(10, 4))
    diff_counts.plot(kind='bar', ax=ax, color='cornflowerblue', edgecolor='white')
    ax.axvline(x=diff_counts.index.tolist().index(0), color='red',
               linestyle='--', label='No seed advantage')
    ax.set_title(f'{label} – Seed Difference Distribution (Winner Seed − Loser Seed)')
    ax.set_xlabel('Seed Difference (Loser − Winner)')
    ax.set_ylabel('Number of games')
    ax.legend()
    plt.tight_layout()
    plt.show()

analyze_seeding_upsets(m_seeds, m_tour_compact, "Men's")
analyze_seeding_upsets(w_seeds, w_tour_compact, "Women's")

## 4. Feature Engineering

We build per-team seasonal features from the **regular season** only (no data leakage):

| Feature | Description |
|---------|-------------|
| `win_pct` | Regular-season win percentage |
| `avg_score` | Average points scored |
| `avg_allowed` | Average points allowed |
| `avg_margin` | Average margin of victory |
| `elo` | Elo rating computed from regular-season results |
| `seed` | Tournament seed (1–16, 0 if not seeded) |
| `massey_rank` | Averaged Massey ordinal rankings (Men's only) |

In [ ]:
# ── 4.1 Basic Team Season Stats ──

def compute_team_season_stats(reg_df: pd.DataFrame) -> pd.DataFrame:
    """Compute per-team, per-season regular-season statistics."""
    if reg_df.empty:
        return pd.DataFrame()

    # Win records
    wins = reg_df.groupby(['Season', 'WTeamID']).agg(
        wins=('WTeamID', 'count'),
        pts_for_w=('WScore', 'mean'),
        pts_against_w=('LScore', 'mean'),
    ).reset_index().rename(columns={'WTeamID': 'TeamID'})

    # Loss records
    losses = reg_df.groupby(['Season', 'LTeamID']).agg(
        losses=('LTeamID', 'count'),
        pts_for_l=('LScore', 'mean'),
        pts_against_l=('WScore', 'mean'),
    ).reset_index().rename(columns={'LTeamID': 'TeamID'})

    stats = wins.merge(losses, on=['Season', 'TeamID'], how='outer').fillna(0)
    stats['games']       = stats['wins'] + stats['losses']
    stats['win_pct']     = stats['wins'] / stats['games'].clip(lower=1)

    # Weighted averages of pts for / against across wins and losses
    stats['avg_score']   = (
        (stats['pts_for_w']    * stats['wins']  +
         stats['pts_for_l']    * stats['losses'])
        / stats['games'].clip(lower=1)
    )
    stats['avg_allowed'] = (
        (stats['pts_against_w'] * stats['wins']  +
         stats['pts_against_l'] * stats['losses'])
        / stats['games'].clip(lower=1)
    )
    stats['avg_margin']  = stats['avg_score'] - stats['avg_allowed']

    return stats[['Season', 'TeamID', 'games', 'wins', 'losses',
                   'win_pct', 'avg_score', 'avg_allowed', 'avg_margin']]


m_stats = compute_team_season_stats(m_reg_compact)
w_stats = compute_team_season_stats(w_reg_compact)

if not m_stats.empty:
    print("Men's stats sample:")
    display(m_stats.head(5))
    print(f"Shape: {m_stats.shape}")

In [ ]:
# ── 4.2 Elo Rating System ──

def compute_elo(reg_df: pd.DataFrame,
                k: float = 20.0,
                initial_elo: float = 1500.0,
                carry_over: float = 0.75) -> pd.DataFrame:
    """
    Compute end-of-regular-season Elo ratings for each team.

    Parameters
    ----------
    reg_df      : regular season compact results
    k           : K-factor (update step)
    initial_elo : default Elo for a new team
    carry_over  : fraction of prior-season Elo to carry forward (rest reverts to initial)
    """
    if reg_df.empty:
        return pd.DataFrame()

    season_end_elo: list = []

    def get_elo(season_elo: dict, season, team, prev_season_elo):
        if (season, team) not in season_elo:
            prior = prev_season_elo.get(team, initial_elo)
            season_elo[(season, team)] = carry_over * prior + (1 - carry_over) * initial_elo
        return season_elo[(season, team)]

    prev_season_elo: dict = {}

    for season in sorted(reg_df['Season'].unique()):
        season_df = reg_df[reg_df['Season'] == season].sort_values('DayNum')
        season_elo: dict = {}

        for _, row in season_df.iterrows():
            w, l = row['WTeamID'], row['LTeamID']
            ew = get_elo(season_elo, season, w, prev_season_elo)
            el = get_elo(season_elo, season, l, prev_season_elo)

            # Margin-of-victory multiplier (optional)
            mov = row['WScore'] - row['LScore']
            mov_mult = np.log1p(abs(mov))

            pw = 1 / (1 + 10 ** ((el - ew) / 400))
            delta = k * mov_mult * (1 - pw)

            season_elo[(season, w)] = ew + delta
            season_elo[(season, l)] = el - delta

        # Save end-of-season Elos
        for (s, t), rating in season_elo.items():
            season_end_elo.append({'Season': s, 'TeamID': t, 'elo': rating})
            prev_season_elo[t] = rating

    return pd.DataFrame(season_end_elo)


m_elo = compute_elo(m_reg_compact)
w_elo = compute_elo(w_reg_compact)

if not m_elo.empty:
    print("Elo sample:")
    display(m_elo.sort_values('elo', ascending=False).head(5))

In [ ]:
# ── 4.3 Seed Features ──

def parse_seeds(seeds_df: pd.DataFrame) -> pd.DataFrame:
    """Extract numeric seed and region from the seed string."""
    if seeds_df.empty:
        return pd.DataFrame()
    df = seeds_df.copy()
    df['Region']  = df['Seed'].str[0]
    df['SeedNum'] = df['Seed'].str.extract(r'(\d+)').astype(int)
    # Play-in flag (seed strings like '11a', '16b')
    df['PlayIn']  = df['Seed'].str.contains(r'[ab]$', regex=True).astype(int)
    return df[['Season', 'TeamID', 'Seed', 'SeedNum', 'Region', 'PlayIn']]


m_seeds_parsed = parse_seeds(m_seeds)
w_seeds_parsed = parse_seeds(w_seeds)

if not m_seeds_parsed.empty:
    print("Seed distribution (Men's):")
    display(m_seeds_parsed['SeedNum'].value_counts().sort_index().to_frame().T)

In [ ]:
# ── 4.4 Massey Ordinal Rankings (Men's only) ──

def compute_massey_avg(massey_df: pd.DataFrame,
                       min_systems: int = 10) -> pd.DataFrame:
    """
    Average Massey ordinal rankings across all ranking systems,
    using the **last snapshot** before the tournament (RankingDayNum ≤ 133).
    """
    if massey_df.empty:
        return pd.DataFrame()

    # Keep only last available ranking per system per team per season
    pre_tourney = massey_df[massey_df['RankingDayNum'] <= 133]
    latest = (pre_tourney
              .sort_values('RankingDayNum')
              .groupby(['Season', 'SystemName', 'TeamID'])
              .last()
              .reset_index())

    # Average rank across systems (lower rank = better)
    avg = (latest.groupby(['Season', 'TeamID'])['OrdinalRank']
                 .agg(['mean', 'count'])
                 .reset_index()
                 .rename(columns={'mean': 'massey_rank', 'count': 'n_systems'}))

    avg = avg[avg['n_systems'] >= min_systems]
    return avg[['Season', 'TeamID', 'massey_rank']]


m_massey_avg = compute_massey_avg(m_massey)
if not m_massey_avg.empty:
    print(f"Massey rankings computed for {m_massey_avg['Season'].nunique()} seasons.")
    display(m_massey_avg.sort_values('massey_rank').head(5))

In [ ]:
# ── 4.5 Merge All Features ──

def build_team_features(stats_df, elo_df, seeds_df, massey_df=None) -> pd.DataFrame:
    """Merge all per-team season features into a single DataFrame."""
    if stats_df.empty:
        return pd.DataFrame()

    feat = stats_df.copy()

    if not elo_df.empty:
        feat = feat.merge(elo_df[['Season', 'TeamID', 'elo']],
                          on=['Season', 'TeamID'], how='left')

    if not seeds_df.empty:
        feat = feat.merge(seeds_df[['Season', 'TeamID', 'SeedNum']],
                          on=['Season', 'TeamID'], how='left')
        feat['SeedNum'] = feat['SeedNum'].fillna(0).astype(int)

    if massey_df is not None and not massey_df.empty:
        feat = feat.merge(massey_df[['Season', 'TeamID', 'massey_rank']],
                          on=['Season', 'TeamID'], how='left')

    return feat


m_feat = build_team_features(m_stats, m_elo, m_seeds_parsed, m_massey_avg)
w_feat = build_team_features(w_stats, w_elo, w_seeds_parsed)

if not m_feat.empty:
    print("Men's feature matrix shape:", m_feat.shape)
    display(m_feat.head(3))

In [ ]:
# ── 4.6 Build Matchup-Level Training Dataset ──

FEATURE_COLS = ['win_pct', 'avg_score', 'avg_allowed', 'avg_margin', 'elo', 'SeedNum']

def build_training_data(tour_df: pd.DataFrame,
                        feat_df: pd.DataFrame,
                        feature_cols: list = FEATURE_COLS) -> tuple:
    """
    Build matchup-level training data from historical tournament results.

    Returns X (features), y (1 = team with lower ID won)
    """
    if tour_df.empty or feat_df.empty:
        return pd.DataFrame(), pd.Series(dtype=float)

    available_cols = [c for c in feature_cols if c in feat_df.columns]

    rows = []
    for _, game in tour_df.iterrows():
        season  = game['Season']
        w_id    = game['WTeamID']
        l_id    = game['LTeamID']

        # Canonical ordering: lower team ID = team A
        t1, t2  = min(w_id, l_id), max(w_id, l_id)
        label   = 1 if w_id == t1 else 0

        f1 = feat_df[(feat_df['Season'] == season) & (feat_df['TeamID'] == t1)]
        f2 = feat_df[(feat_df['Season'] == season) & (feat_df['TeamID'] == t2)]

        if f1.empty or f2.empty:
            continue

        f1 = f1.iloc[0]
        f2 = f2.iloc[0]

        row = {'Season': season, 'T1': t1, 'T2': t2, 'label': label}
        for col in available_cols:
            row[f'T1_{col}'] = f1.get(col, np.nan)
            row[f'T2_{col}'] = f2.get(col, np.nan)
            row[f'diff_{col}'] = f1.get(col, np.nan) - f2.get(col, np.nan)

        rows.append(row)

    train_df = pd.DataFrame(rows)
    meta_cols = ['Season', 'T1', 'T2', 'label']
    xcols = [c for c in train_df.columns if c not in meta_cols]

    X = train_df[xcols].fillna(0)
    y = train_df['label']
    return X, y


X_m, y_m = build_training_data(m_tour_compact, m_feat)
X_w, y_w = build_training_data(w_tour_compact, w_feat)

print(f"Men's training set   : X={X_m.shape}, positives={y_m.mean():.3f}")
print(f"Women's training set : X={X_w.shape}, positives={y_w.mean():.3f}")

In [ ]:
# ── Feature correlation heatmap ──
def plot_feature_correlations(X: pd.DataFrame, label: str):
    if X.empty:
        return
    diff_cols = [c for c in X.columns if c.startswith('diff_')]
    if not diff_cols:
        return
    fig, ax = plt.subplots(figsize=(8, 6))
    corr = X[diff_cols].corr()
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
                linewidths=0.5, vmin=-1, vmax=1)
    ax.set_title(f'{label} – Feature Correlation (difference features)')
    plt.tight_layout()
    plt.show()

plot_feature_correlations(X_m, "Men's")
plot_feature_correlations(X_w, "Women's")

## 5. Model Training

We train three model families and evaluate with 5-fold cross-validated log-loss:
1. **Logistic Regression** – interpretable baseline
2. **XGBoost** – gradient-boosted trees (if available)
3. **LightGBM** – fast gradient boosting (if available)

In [ ]:
def evaluate_model(model, X: pd.DataFrame, y: pd.Series,
                   label: str, n_folds: int = N_FOLDS) -> float:
    """Cross-validate a model and report mean log-loss."""
    if X.empty or len(y) == 0:
        print(f"[{label}] No data – skipping.")
        return np.nan
    cv = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    scores = cross_val_score(model, X, y,
                             scoring='neg_log_loss', cv=cv)
    mean_ll = -scores.mean()
    std_ll  = scores.std()
    print(f"[{label}] CV Log-Loss = {mean_ll:.5f} ± {std_ll:.5f}")
    return mean_ll

In [ ]:
# ── 5.1 Logistic Regression ──
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=1.0, max_iter=1000, random_state=SEED))
])

ll_lr_m = evaluate_model(lr_pipeline, X_m, y_m, 'LR Men\'s')
ll_lr_w = evaluate_model(lr_pipeline, X_w, y_w, 'LR Women\'s')

In [ ]:
# ── 5.2 XGBoost ──
if HAS_XGB:
    xgb_model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=SEED,
        verbosity=0,
    )
    ll_xgb_m = evaluate_model(xgb_model, X_m, y_m, 'XGB Men\'s')
    ll_xgb_w = evaluate_model(xgb_model, X_w, y_w, 'XGB Women\'s')
else:
    ll_xgb_m = ll_xgb_w = np.nan

In [ ]:
# ── 5.3 LightGBM ──
if HAS_LGB:
    lgb_model = lgb.LGBMClassifier(
        n_estimators=300,
        num_leaves=31,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=SEED,
        verbose=-1,
    )
    ll_lgb_m = evaluate_model(lgb_model, X_m, y_m, 'LGB Men\'s')
    ll_lgb_w = evaluate_model(lgb_model, X_w, y_w, 'LGB Women\'s')
else:
    ll_lgb_m = ll_lgb_w = np.nan

In [ ]:
# ── Model comparison plot ──
results = pd.DataFrame({
    'Model':    ['Logistic Reg', 'XGBoost', 'LightGBM'],
    "Men's LL":  [ll_lr_m, ll_xgb_m, ll_lgb_m],
    "Women's LL":[ll_lr_w, ll_xgb_w, ll_lgb_w],
}).set_index('Model')

ax = results.plot(kind='bar', figsize=(8, 4), rot=0, edgecolor='white')
ax.set_ylabel('CV Log-Loss (lower = better)')
ax.set_title('Model Comparison – Cross-Validated Log-Loss')
ax.axhline(y=0.693, color='red', linestyle='--', alpha=0.6, label='Random (0.693)')
ax.legend()
plt.tight_layout()
plt.show()

print("\nResults Summary:")
display(results)

## 6. Ensemble & Calibration

We create a simple **weighted average ensemble** over the models and use Platt scaling (sigmoid calibration) to ensure well-calibrated probabilities.

In [ ]:
def train_final_model(X: pd.DataFrame, y: pd.Series,
                      use_xgb: bool = HAS_XGB,
                      use_lgb: bool = HAS_LGB):
    """Train an ensemble of models on the full dataset and return a callable."""
    if X.empty or len(y) == 0:
        return None

    models = []
    weights = []

    # Logistic Regression (always available)
    lr = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=SEED))
    ])
    lr.fit(X, y)
    models.append(lr)
    weights.append(1.0)

    if use_xgb:
        xgb_m = xgb.XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            use_label_encoder=False, eval_metric='logloss',
            random_state=SEED, verbosity=0,
        )
        xgb_m.fit(X, y)
        models.append(xgb_m)
        weights.append(1.5)

    if use_lgb:
        lgb_m = lgb.LGBMClassifier(
            n_estimators=300, num_leaves=31, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            random_state=SEED, verbose=-1,
        )
        lgb_m.fit(X, y)
        models.append(lgb_m)
        weights.append(1.5)

    total_w = sum(weights)

    def predict_proba_ensemble(X_new: pd.DataFrame) -> np.ndarray:
        probs = np.zeros(len(X_new))
        for m, w in zip(models, weights):
            probs += (w / total_w) * m.predict_proba(X_new)[:, 1]
        return probs

    return predict_proba_ensemble


print("Training final Men's ensemble...")
m_predict = train_final_model(X_m, y_m)
print("Training final Women's ensemble...")
w_predict = train_final_model(X_w, y_w)
print("Done.")

## 7. Submission Generation

For each possible matchup in the 2026 tournament (from the sample submission file), we:
1. Parse the team IDs and season from the submission ID
2. Look up team features for the 2026 season
3. Predict win probabilities using the ensemble
4. Clip to [0.05, 0.95] to avoid extreme log-loss
5. Save to `submission.csv`

In [ ]:
def build_submission_features(sample_df: pd.DataFrame,
                               feat_df: pd.DataFrame,
                               feature_cols: list = FEATURE_COLS
                               ) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build feature matrix for all matchups in the sample submission.

    Returns (meta_df, X_sub) where meta_df contains the original ID column.
    """
    if sample_df.empty or feat_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    available_cols = [c for c in feature_cols if c in feat_df.columns]

    # Parse IDs
    split = sample_df['ID'].str.split('_', expand=True)
    sample_df = sample_df.copy()
    sample_df['Season'] = split[0].astype(int)
    sample_df['T1']     = split[1].astype(int)
    sample_df['T2']     = split[2].astype(int)

    rows = []
    for _, row in sample_df.iterrows():
        season = row['Season']
        t1, t2 = row['T1'], row['T2']

        f1 = feat_df[(feat_df['Season'] == season) & (feat_df['TeamID'] == t1)]
        f2 = feat_df[(feat_df['Season'] == season) & (feat_df['TeamID'] == t2)]

        r = {}
        for col in available_cols:
            v1 = f1.iloc[0][col] if not f1.empty else 0.0
            v2 = f2.iloc[0][col] if not f2.empty else 0.0
            r[f'T1_{col}']   = v1
            r[f'T2_{col}']   = v2
            r[f'diff_{col}'] = v1 - v2
        rows.append(r)

    X_sub = pd.DataFrame(rows).fillna(0)
    return sample_df[['ID', 'Season', 'T1', 'T2']], X_sub


print("Building Men's submission features...")
m_meta, X_m_sub = build_submission_features(m_sample_sub, m_feat)
print(f"  Men's: {len(m_meta):,} matchups")

print("Building Women's submission features...")
w_meta, X_w_sub = build_submission_features(w_sample_sub, w_feat)
print(f"  Women's: {len(w_meta):,} matchups")

In [ ]:
def generate_submission(predict_fn,
                        meta_df: pd.DataFrame,
                        X_sub: pd.DataFrame,
                        clip_lo: float = CLIP_LO,
                        clip_hi: float = CLIP_HI) -> pd.DataFrame:
    """Run predictions and assemble submission DataFrame."""
    if predict_fn is None or meta_df.empty or X_sub.empty:
        # Fall back to 0.5 (no information)
        if not meta_df.empty:
            return meta_df[['ID']].assign(Pred=0.5)
        return pd.DataFrame(columns=['ID', 'Pred'])

    preds = predict_fn(X_sub)
    preds = np.clip(preds, clip_lo, clip_hi)
    return meta_df[['ID']].assign(Pred=preds)


sub_m = generate_submission(m_predict, m_meta, X_m_sub)
sub_w = generate_submission(w_predict, w_meta, X_w_sub)

# Combine both genders
submission = pd.concat([sub_m, sub_w], ignore_index=True)
print(f"Total submission rows: {len(submission):,}")
display(submission.head(10))

In [ ]:
# ── Sanity checks ──
print("=== Submission Sanity Checks ===")
print(f"Rows         : {len(submission):,}")
print(f"Pred range   : [{submission['Pred'].min():.4f}, {submission['Pred'].max():.4f}]")
print(f"Pred mean    : {submission['Pred'].mean():.4f}  (expect ~0.50)")
print(f"Null preds   : {submission['Pred'].isna().sum()}")
print(f"Duplicate IDs: {submission['ID'].duplicated().sum()}")

# Distribution plot
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(submission['Pred'], bins=50, color='steelblue', edgecolor='white')
ax.set_title('Prediction Distribution')
ax.set_xlabel('Predicted Probability (Team 1 wins)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# ── Save submission ──
out_path = os.path.join(OUTPUT_DIR, 'submission.csv')
submission.to_csv(out_path, index=False)
print(f"Submission saved → {out_path}")
display(submission.head())

---
## Appendix A – Advanced Feature Ideas

The features above are a strong starting point. Below are additional ideas that can further improve model performance:

### A.1 Advanced Box-Score Metrics
Using `*DetailedResults.csv` files:
- **Offensive Efficiency** = Points / Possessions
- **Defensive Efficiency** = Opponent Points / Possessions
- **Possessions** ≈ FGA – OREB + TOV + 0.44×FTA
- **Effective FG%** = (FGM + 0.5×FGM3) / FGA
- **Turnover rate**, **Free-throw rate**, **Rebound rate**

### A.2 Strength of Schedule
- Average Elo of opponents faced during regular season
- Conference strength (based on historical tournament performance)
- Non-conference record vs. high-rated opponents

### A.3 Momentum / Recency Features
- Win rate in last N games (e.g., N=10)
- Trend in point differential over the season
- Performance in conference tournament

### A.4 Historical Tournament Success
- Team's historical tournament win rate at a given seed
- Number of Final Four appearances in past 10 years
- Head-to-head tournament record between matchup teams

### A.5 Location / Travel Advantage
- Distance from team's home arena to game location
- Regional bracket alignment (teams often play closer to home in early rounds)

### A.6 Roster & Coaching
- Returning starters percentage (if available)
- Coach's tournament win rate
- Transfer portal impact

## Appendix B – Alternative Modelling Approaches

### B.1 Bradley-Terry Model
A classic paired comparison model. Each team has a latent strength parameter $\lambda_i$:
$$P(i \text{ beats } j) = \frac{\lambda_i}{\lambda_i + \lambda_j}$$
Parameters are estimated via maximum likelihood from historical results.

### B.2 Neural Network
A shallow MLP with batch-normalisation and dropout:
```
Input (team features) → Dense(64, ReLU) → BatchNorm → Dropout(0.3)
                      → Dense(32, ReLU) → Dense(1, Sigmoid)
```
Siamese-style networks (sharing weights between the two teams) ensure the model is properly symmetric.

### B.3 Bayesian Approaches
- **Bayesian Elo**: treat Elo as a probabilistic model with uncertainty estimates
- **Gaussian Process**: non-parametric model capturing team strength uncertainty

### B.4 Seed-Only Baseline
A strong sanity-check baseline: simply predict the higher-seeded team wins with a fixed probability derived from historical upset rates:

| Matchup | Historical Win % (fav) |
|---------|------------------------|
| 1 vs 16 | 99.3% |
| 2 vs 15 | 94.1% |
| 3 vs 14 | 85.0% |
| 4 vs 13 | 78.8% |
| 5 vs 12 | 64.6% |
| 6 vs 11 | 62.5% |
| 7 vs 10 | 60.7% |
| 8 vs 9  | 50.9% |

---
## Summary

| Step | Description | Status |
|------|-------------|--------|
| Data Loading | All CSV files loaded from `DATA_DIR` | ✅ |
| EDA | Score distributions, seeding upsets analysed | ✅ |
| Feature Engineering | Win %, Elo, seed, Massey rankings | ✅ |
| Model Training | Logistic Regression, XGBoost, LightGBM | ✅ |
| Ensemble | Weighted average of all models | ✅ |
| Submission | Clipped predictions saved to `submission.csv` | ✅ |

### Next Steps
- Add detailed box-score features (offensive/defensive efficiency)
- Hyper-parameter tuning via Optuna
- Explore Neural Network / Bradley-Terry models
- Incorporate strength-of-schedule adjustments
- Analyse 2026-specific team data as it becomes available